
# **Case Study: Pneumonia Detection Using Fine-Tuned CNN (ResNet50)**
## **Objective**
The goal of this case study is to develop a deep learning model capable of classifying **chest X-ray images** into two categories:  
1. **Normal** (No Pneumonia)  
2. **Pneumonia** (Infected Lungs)  

To achieve this, we use **transfer learning** with the **ResNet50** model, which has been pre-trained on the **ImageNet** dataset. Since medical images differ significantly from natural images, fine-tuning the model helps improve accuracy.

## **Approach**
### **Step 1: Feature Extraction**
- We **freeze** all layers of ResNet50 and use it as a **feature extractor**.
- A **custom classification head** is added on top of ResNet50.
- The model is **trained for 10 epochs** using chest X-ray images.

### **Step 2: Fine-Tuning**
- We **unfreeze the last 50 layers** of ResNet50.
- The model is **fine-tuned for 5 more epochs** with a **lower learning rate** to adapt to medical image features.
- Fine-tuning allows the deeper layers to learn pneumonia-specific patterns.

## **Why Fine-Tuning?**
Fine-tuning is necessary because:
1. Medical images have **different textures** than ImageNet images (which mostly contain natural objects).
2. Some **lower-level features** from ImageNet are useful, but deeper layers must be trained to recognize pneumonia patterns.
3. A **lower learning rate** ensures that the pre-trained knowledge is not lost but adapted.

## **Dataset**
The dataset consists of **chest X-ray images** collected from real-world medical studies. The images are split into:
- **Training Set (80%)**
- **Validation Set (20%)**

The dataset is loaded using **ImageDataGenerator**, which also applies **data augmentation** (rotation, shifting, flipping) to improve generalization.

## **Conclusion**
By leveraging **pre-trained models and fine-tuning**, we enhance model performance for pneumonia detection while reducing training time and dataset size requirements.
"""

## **Imports & Model Loading**

In [1]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load pre-trained model (ResNet50) without top layers
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 56s 1us/step


## **Feature Extraction & Custom Head**

In [11]:
# Freeze the base model layers (Feature Extraction)
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(2, activation='softmax')(x)  # Apply the Dense layer to x to get the output tensor

## **Model Compilation & Summary**

In [14]:
# Create new model
model = Model(inputs=base_model.input, outputs=outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print model summary
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 24,768,642 (94.48 MB)

 Trainable params: 1,180,930 (4.50 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

# **Data Augmentation & Data Loading**

In [21]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load training and validation data
train_generator = train_datagen.flow_from_directory(
    r"C:\Users\salah\OneDrive\Desktop\Chest X-Ray\train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    r"C:\Users\salah\OneDrive\Desktop\Chest X-Ray\val",
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 4173 images belonging to 2 classes.
Found 2 images belonging to 2 classes.


# **Feature Extraction Training**

In [26]:
# Train the model (Feature Extraction)
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=1,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)


131/131 ━━━━━━━━━━━━━━━━━━━━ 278s 2s/step - accuracy: 0.7331 - loss: 0.6076 - val_accuracy: 0.5000 - val_loss: 0.7602


# **Fine-Tuning Setup**

In [32]:
# Unfreeze the last 50 layers for fine-tuning
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Recompile model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


# **Fine-Tuning Training**

In [ ]:
# Fine-tune the model
model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)


Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 335s 2s/step - accuracy: 0.8882 - loss: 0.2634 - val_accuracy: 0.5000 - val_loss: 19.4056
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 319s 2s/step - accuracy: 0.8883 - loss: 0.2507 - val_accuracy: 0.5000 - val_loss: 10.1426
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 318s 2s/step - accuracy: 0.9066 - loss: 0.2327 - val_accuracy: 0.5000 - val_loss: 39.3914
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 333s 3s/step - accuracy: 0.8991 - loss: 0.2421 - val_accuracy: 0.5000 - val_loss: 21.3406
Epoch 5/5
102/131 ━━━━━━━━━━━━━━━━━━━━ 1:11 2s/step - accuracy: 0.8921 - loss: 0.2563

In [ ]:
# Save the fine-tuned model
model.save('fine_tuned_pneumonia_model.h5')
